In [ ]:
# === Cell 0: Pip Installs (uncomment as needed) ===
# !pip install yfinance TA-Lib numpy pandas vectorbt scipy ccxt
# !pip install matplotlib

In [ ]:
# === Cell 1: Colab Repo Clone ===
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
# === Cell 2: Imports ===
import sys, os
import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from itertools import product
import time

sys.path.insert(0, os.path.join(os.getcwd(), 'lib'))
from data_manager import *

warnings.filterwarnings('ignore')

print(f"vectorbt {vbt.__version__} | numpy {np.__version__} | pandas {pd.__version__}")

# Crypto Intraday Grid Search (GPU-Accelerated)
- Tests MACD, Donchian, RSI Mean-Reversion, and Stochastic on 1H/4H timeframes
- Massive parameter grids (10,000+ combos per strategy)
- Includes stop-loss and take-profit as grid dimensions
- Regime filter overlay (volatility-based)
- Designed for GPU/Colab Pro execution

In [ ]:
# === Cell 4: Configuration ===

# === DATA CONFIG ===
TICKERS = ['BTC-USD', 'ETH-USD', 'XRP-USD', 'SOL-USD']
TIMEFRAME = '1h'  # '1h' or '4h'
START_DATE = '2022-01-01'  # Intraday data availability is limited
TRAIN_RATIO = 0.60

# === BACKTEST CONFIG ===
INIT_CASH = 100_000
FEES = 0.0005
SLIPPAGE = 0.0005

# === STRATEGY SELECTION ===
STRATEGIES = ['macd', 'donchian', 'rsi_mr', 'stochastic']

# === PARAMETER GRIDS (massive — designed for GPU) ===
PARAM_GRIDS = {
    'macd': {
        'fast': range(5, 50, 3),
        'slow': range(15, 100, 5),
        'signal': range(2, 20, 2),
    },
    'donchian': {
        'entry': range(5, 60, 3),
        'exit': range(2, 25, 2),
        'filter': range(10, 100, 5),
    },
    'rsi_mr': {
        'rsi_len': range(5, 30, 2),
        'oversold': range(15, 40, 5),
        'overbought': range(60, 90, 5),
    },
    'stochastic': {
        'k_period': range(5, 25, 2),
        'd_period': range(2, 10, 2),
        'oversold': range(10, 35, 5),
        'overbought': range(65, 95, 5),
    },
}

# === RISK MANAGEMENT GRID (applied to ALL strategies) ===
SL_PCTS = [None, 0.01, 0.02, 0.03, 0.05]  # Stop-loss as % of entry price
TP_PCTS = [None, 0.02, 0.04, 0.06, 0.10]  # Take-profit as % of entry price

# === REGIME FILTER ===
USE_REGIME_FILTER = True
VOLATILITY_LOOKBACK = 20  # bars for ATR calculation
VOLATILITY_THRESHOLD = 1.5  # trade only when ATR ratio > threshold (trending)

# Print grid sizes
for strat, grid in PARAM_GRIDS.items():
    n = 1
    for v in grid.values():
        n *= len(list(v))
    n_risk = len(SL_PCTS) * len(TP_PCTS)
    print(f"{strat.upper()}: {n} param combos x {n_risk} risk combos = {n * n_risk:,} total")

In [ ]:
# === Cell 5: Data Download ===
# DOWNLOAD INTRADAY DATA
# For 1h data: yfinance supports ~730 days of hourly data
# For longer history: use ccxt with Coinbase/Binance

import ccxt

def download_crypto_intraday(ticker, timeframe='1h', start_date='2022-01-01', source='yfinance'):
    """Download intraday crypto data. Falls back to ccxt if yfinance doesn't have enough history."""
    if source == 'yfinance':
        # yfinance intraday
        interval_map = {'1h': '1h', '4h': '1h', '15m': '15m', '5m': '5m'}
        period_map = {'1h': '730d', '4h': '730d', '15m': '60d', '5m': '60d'}
        df = yf.download(ticker, period=period_map.get(timeframe, '730d'),
                        interval=interval_map.get(timeframe, '1h'), progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        if timeframe == '4h' and interval_map.get(timeframe) == '1h':
            # Resample 1h to 4h
            df = df.resample('4h').agg({
                'Open': 'first', 'High': 'max', 'Low': 'min',
                'Close': 'last', 'Volume': 'sum'
            }).dropna()
        return df
    
    elif source == 'ccxt':
        # Use ccxt for longer history (requires no API key for public data)
        exchange = ccxt.coinbase()
        symbol = ticker.replace('-', '/')  # BTC-USD -> BTC/USD
        tf_map = {'1h': '1h', '4h': '4h', '15m': '15m', '1m': '1m'}
        
        since = exchange.parse8601(f"{start_date}T00:00:00Z")
        all_candles = []
        while True:
            candles = exchange.fetch_ohlcv(symbol, tf_map.get(timeframe, '1h'),
                                           since=since, limit=1000)
            if not candles:
                break
            all_candles.extend(candles)
            since = candles[-1][0] + 1
            if len(candles) < 1000:
                break
        
        df = pd.DataFrame(all_candles, columns=['timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        df.set_index('timestamp', inplace=True)
        return df

# Download for all tickers
data = {}
for ticker in TICKERS:
    try:
        df = download_crypto_intraday(ticker, TIMEFRAME, START_DATE, source='yfinance')
        if df.empty:
            print(f"{ticker}: yfinance empty, trying ccxt...")
            df = download_crypto_intraday(ticker, TIMEFRAME, START_DATE, source='ccxt')
    except Exception as e:
        print(f"{ticker}: yfinance failed ({e}), trying ccxt...")
        try:
            df = download_crypto_intraday(ticker, TIMEFRAME, START_DATE, source='ccxt')
        except Exception as e2:
            print(f"{ticker}: BOTH SOURCES FAILED - {e2}")
            continue
    
    close = df['Close'].astype(float).squeeze()
    high = df['High'].astype(float).squeeze()
    low = df['Low'].astype(float).squeeze()
    close.name = 'price'
    data[ticker] = {'close': close, 'high': high, 'low': low, 'df': df}
    print(f"{ticker}: {len(close)} bars ({TIMEFRAME}) from {close.index[0]} to {close.index[-1]}")

# Align to common range
if len(data) > 1:
    common_start = max(d['close'].index[0] for d in data.values())
    common_end = min(d['close'].index[-1] for d in data.values())
    for ticker in list(data.keys()):
        mask = (data[ticker]['close'].index >= common_start) & (data[ticker]['close'].index <= common_end)
        for k in ['close', 'high', 'low']:
            data[ticker][k] = data[ticker][k][mask]
    print(f"\nAligned: {common_start} to {common_end}")

print(f"Loaded {len(data)} tickers")

In [ ]:
# === Cell 6: Signal Generation Functions with Regime Filter ===
# SIGNAL FUNCTIONS (all with 1-bar execution delay)

def calc_regime_filter(close, high, low, lookback=20, threshold=1.5):
    """Volatility regime filter: True when market is trending (high ATR relative to recent)."""
    atr = pd.Series(talib.ATR(high.values, low.values, close.values, timeperiod=lookback), index=close.index)
    atr_ma = atr.rolling(lookback * 3).mean()
    regime = (atr / atr_ma) > threshold  # high vol = trending
    return regime.shift(1).fillna(False)  # shift to avoid lookahead

def macd_signals(close, fast, slow, signal, regime=None):
    ml, sl, _ = talib.MACD(close.values.astype(float), fastperiod=fast, slowperiod=slow, signalperiod=signal)
    macd_s = pd.Series(ml, index=close.index)
    sig_s = pd.Series(sl, index=close.index)
    entries_raw = (macd_s.shift(1) <= sig_s.shift(1)) & (macd_s > sig_s)
    exits_raw = (macd_s.shift(1) >= sig_s.shift(1)) & (macd_s < sig_s)
    if regime is not None:
        entries_raw = entries_raw & regime
    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)), index=close.index, dtype=bool)
    exits = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)), index=close.index, dtype=bool)
    return entries, exits

def donchian_signals(close, high, low, entry_period, exit_period, filter_period, regime=None):
    upper = pd.Series(talib.MAX(high.values, entry_period), index=close.index).shift(1)
    lower = pd.Series(talib.MIN(low.values, exit_period), index=close.index).shift(1)
    trend = pd.Series(talib.SMA(close.values, filter_period), index=close.index).shift(1)
    entries_raw = (close > upper) & (close > trend)
    exits_raw = (close < lower)
    if regime is not None:
        entries_raw = entries_raw & regime
    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)), index=close.index, dtype=bool)
    exits = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)), index=close.index, dtype=bool)
    return entries, exits

def rsi_mr_signals(close, rsi_len, oversold, overbought, regime=None):
    rsi = pd.Series(talib.RSI(close.values, timeperiod=rsi_len), index=close.index)
    entries_raw = (rsi.shift(1) <= oversold) & (rsi > oversold)
    exits_raw = (rsi.shift(1) <= overbought) & (rsi > overbought)
    if regime is not None:
        # For mean-reversion: trade when NOT trending (low vol regime)
        entries_raw = entries_raw & ~regime
    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)), index=close.index, dtype=bool)
    exits = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)), index=close.index, dtype=bool)
    return entries, exits

def stochastic_signals(close, high, low, k_period, d_period, oversold, overbought, regime=None):
    slowk, slowd = talib.STOCH(high.values, low.values, close.values,
                                fastk_period=k_period, slowk_period=d_period,
                                slowk_matype=0, slowd_period=d_period, slowd_matype=0)
    k = pd.Series(slowk, index=close.index)
    d = pd.Series(slowd, index=close.index)
    entries_raw = (k.shift(1) <= oversold) & (k > oversold) & (k > d)
    exits_raw = (k.shift(1) <= overbought) & (k > overbought)
    if regime is not None:
        entries_raw = entries_raw & ~regime  # mean-reversion: low vol
    entries = pd.Series(np.where(entries_raw.shift(1).isna(), False, entries_raw.shift(1)), index=close.index, dtype=bool)
    exits = pd.Series(np.where(exits_raw.shift(1).isna(), False, exits_raw.shift(1)), index=close.index, dtype=bool)
    return entries, exits

def apply_sl_tp(close, entries, exits, sl_pct, tp_pct):
    """Apply stop-loss and take-profit overlays to existing signals."""
    if sl_pct is None and tp_pct is None:
        return entries, exits
    
    new_exits = exits.copy()
    entry_price = np.nan
    in_position = False
    
    for i in range(len(close)):
        if entries.iloc[i] and not in_position:
            entry_price = close.iloc[i]
            in_position = True
        elif in_position:
            pct_change = (close.iloc[i] - entry_price) / entry_price
            if sl_pct is not None and pct_change <= -sl_pct:
                new_exits.iloc[i] = True
                in_position = False
            elif tp_pct is not None and pct_change >= tp_pct:
                new_exits.iloc[i] = True
                in_position = False
            elif exits.iloc[i]:
                in_position = False
    
    return entries, new_exits

print("Signal functions with regime filter + SL/TP defined.")

In [ ]:
# === Cell 7: GPU-Accelerated Grid Search Engine ===

from itertools import product
import time

def run_grid_search(ticker, strategy, close, high, low, param_grid, sl_pcts, tp_pcts,
                    train_ratio=0.60, regime=None, min_trades=10):
    """Run massive grid search for a single ticker + strategy combo."""
    split_idx = int(len(close) * train_ratio)
    train_close = close.iloc[:split_idx]
    train_high = high.iloc[:split_idx] if high is not None else None
    train_low = low.iloc[:split_idx] if low is not None else None
    train_regime = regime.iloc[:split_idx] if regime is not None else None
    
    param_names = list(param_grid.keys())
    param_values = [list(v) for v in param_grid.values()]
    all_combos = list(product(*param_values))
    
    results = []
    t0 = time.time()
    total = len(all_combos) * len(sl_pcts) * len(tp_pcts)
    done = 0
    
    for combo in all_combos:
        params = dict(zip(param_names, combo))
        
        # Skip invalid combos
        if strategy == 'macd' and params['fast'] >= params['slow']:
            done += len(sl_pcts) * len(tp_pcts)
            continue
        
        try:
            if strategy == 'macd':
                entries, exits = macd_signals(train_close, params['fast'], params['slow'],
                                              params['signal'], train_regime)
            elif strategy == 'donchian':
                entries, exits = donchian_signals(train_close, train_high, train_low,
                                                  params['entry'], params['exit'],
                                                  params['filter'], train_regime)
            elif strategy == 'rsi_mr':
                if params['oversold'] >= params['overbought']:
                    done += len(sl_pcts) * len(tp_pcts)
                    continue
                entries, exits = rsi_mr_signals(train_close, params['rsi_len'],
                                                params['oversold'], params['overbought'], train_regime)
            elif strategy == 'stochastic':
                if params['oversold'] >= params['overbought']:
                    done += len(sl_pcts) * len(tp_pcts)
                    continue
                entries, exits = stochastic_signals(train_close, train_high, train_low,
                                                    params['k_period'], params['d_period'],
                                                    params['oversold'], params['overbought'], train_regime)
        except Exception:
            done += len(sl_pcts) * len(tp_pcts)
            continue
        
        for sl in sl_pcts:
            for tp in tp_pcts:
                done += 1
                try:
                    e, x = apply_sl_tp(train_close, entries, exits, sl, tp)
                    pf = vbt.Portfolio.from_signals(
                        close=train_close, entries=e, exits=x,
                        init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=TIMEFRAME
                    )
                    
                    n_trades = len(pf.trades)
                    if n_trades < min_trades:
                        continue
                    
                    sharpe = float(pf.sharpe_ratio(freq=TIMEFRAME))
                    if np.isnan(sharpe) or np.isinf(sharpe):
                        continue
                    
                    results.append({
                        **params,
                        'sl_pct': sl, 'tp_pct': tp,
                        'sharpe': sharpe,
                        'return': float(pf.total_return()),
                        'max_dd': float(pf.max_drawdown()),
                        'trades': n_trades,
                        'win_rate': float((pf.trades.returns.values > 0).sum() / n_trades * 100) if n_trades > 0 else 0,
                    })
                except Exception:
                    continue
                
                if done % 5000 == 0:
                    elapsed = time.time() - t0
                    rate = done / elapsed if elapsed > 0 else 0
                    eta = (total - done) / rate / 60 if rate > 0 else 0
                    print(f"  {done:,}/{total:,} ({done/total*100:.0f}%) | "
                          f"{rate:.0f} combos/sec | ETA: {eta:.1f}min | "
                          f"Valid: {len(results)}")
    
    elapsed = time.time() - t0
    print(f"  Done: {len(results):,} valid results from {total:,} combos in {elapsed:.1f}s")
    
    return pd.DataFrame(results).sort_values('sharpe', ascending=False) if results else pd.DataFrame()

print("Grid search engine ready.")

In [ ]:
# === Cell 8: Run Full Grid Search ===

all_results = {}

for ticker in TICKERS:
    if ticker not in data:
        print(f"Skipping {ticker} — no data")
        continue
    
    close = data[ticker]['close']
    high = data[ticker]['high']
    low = data[ticker]['low']
    
    # Calculate regime filter
    regime = None
    if USE_REGIME_FILTER:
        regime = calc_regime_filter(close, high, low, VOLATILITY_LOOKBACK, VOLATILITY_THRESHOLD)
        pct_trending = regime.sum() / len(regime) * 100
        print(f"\n{ticker}: Regime filter — {pct_trending:.1f}% of bars are 'trending'")
    
    for strategy in STRATEGIES:
        key = f"{strategy}_{ticker}"
        print(f"\n{'='*60}")
        print(f"GRID SEARCH: {strategy.upper()} on {ticker} ({TIMEFRAME})")
        print(f"{'='*60}")
        
        grid = PARAM_GRIDS[strategy]
        results_df = run_grid_search(
            ticker, strategy, close, high, low, grid,
            SL_PCTS, TP_PCTS, TRAIN_RATIO, regime
        )
        
        if not results_df.empty:
            all_results[key] = results_df
            print(f"\nTop 5 for {key}:")
            print(results_df.head().to_string(index=False))
        else:
            print(f"  No valid results for {key}")

print(f"\n{'='*60}")
print(f"Grid search complete. {len(all_results)} strategy-ticker combos with results.")

In [ ]:
# === Cell 9: OOS Validation of Top Performers ===
# OUT-OF-SAMPLE VALIDATION — TOP 5 PER COMBO

oos_results = []

for key, res_df in all_results.items():
    strategy, ticker = key.split('_', 1)
    close = data[ticker]['close']
    high = data[ticker]['high']
    low = data[ticker]['low']
    split_idx = int(len(close) * TRAIN_RATIO)
    val_close = close.iloc[split_idx:]
    val_high = high.iloc[split_idx:]
    val_low = low.iloc[split_idx:]
    
    regime = None
    if USE_REGIME_FILTER:
        regime = calc_regime_filter(close, high, low, VOLATILITY_LOOKBACK, VOLATILITY_THRESHOLD)
        val_regime = regime.iloc[split_idx:]
    else:
        val_regime = None
    
    top5 = res_df.head(5)
    
    for idx, row in top5.iterrows():
        params = {k: row[k] for k in PARAM_GRIDS[strategy].keys()}
        int_params = {k: int(v) for k, v in params.items()}
        sl = row.get('sl_pct', None)
        tp = row.get('tp_pct', None)
        if pd.isna(sl): sl = None
        if pd.isna(tp): tp = None
        
        try:
            if strategy == 'macd':
                entries, exits = macd_signals(val_close, int_params['fast'], int_params['slow'],
                                              int_params['signal'], val_regime)
            elif strategy == 'donchian':
                entries, exits = donchian_signals(val_close, val_high, val_low,
                                                  int_params['entry'], int_params['exit'],
                                                  int_params['filter'], val_regime)
            elif strategy == 'rsi_mr':
                entries, exits = rsi_mr_signals(val_close, int_params['rsi_len'],
                                                int_params['oversold'], int_params['overbought'], val_regime)
            elif strategy == 'stochastic':
                entries, exits = stochastic_signals(val_close, val_high, val_low,
                                                    int_params['k_period'], int_params['d_period'],
                                                    int_params['oversold'], int_params['overbought'], val_regime)
            
            e, x = apply_sl_tp(val_close, entries, exits, sl, tp)
            pf = vbt.Portfolio.from_signals(
                close=val_close, entries=e, exits=x,
                init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=TIMEFRAME
            )
            
            n_trades = len(pf.trades)
            oos_sharpe = float(pf.sharpe_ratio(freq=TIMEFRAME))
            
            oos_results.append({
                'strategy': strategy, 'ticker': ticker,
                **int_params,
                'sl_pct': sl, 'tp_pct': tp,
                'is_sharpe': float(row['sharpe']),
                'oos_sharpe': oos_sharpe,
                'sharpe_degradation': (float(row['sharpe']) - oos_sharpe) / abs(float(row['sharpe'])) if abs(float(row['sharpe'])) > 0.01 else 0,
                'is_return': float(row['return']),
                'oos_return': float(pf.total_return()),
                'is_maxdd': float(row['max_dd']),
                'oos_maxdd': float(pf.max_drawdown()),
                'is_trades': int(row['trades']),
                'oos_trades': n_trades,
                'oos_win_rate': float((pf.trades.returns.values > 0).sum() / n_trades * 100) if n_trades > 0 else 0,
            })
        except Exception as e:
            print(f"  OOS validation failed for {key}: {e}")
            continue

oos_df = pd.DataFrame(oos_results).sort_values('oos_sharpe', ascending=False)

print("=" * 80)
print(f"OOS VALIDATION RESULTS — {len(oos_df)} combos validated")
print("=" * 80)
print(f"\n{'Strategy':<12} {'Ticker':<10} {'IS Sharpe':>10} {'OOS Sharpe':>10} {'Degradation':>12} {'OOS Ret':>10} {'OOS DD':>10} {'Trades':>7}")
print("-" * 81)
for _, r in oos_df.head(20).iterrows():
    deg_flag = "OVERFIT" if r['sharpe_degradation'] > 0.5 else "OK" if r['sharpe_degradation'] < 0.3 else "CAUTION"
    print(f"{r['strategy']:<12} {r['ticker']:<10} {r['is_sharpe']:>10.3f} {r['oos_sharpe']:>10.3f} "
          f"{r['sharpe_degradation']:>10.1%} {deg_flag:>3} {r['oos_return']:>9.1%} {r['oos_maxdd']:>9.1%} {r['oos_trades']:>7}")

# Flag best non-overfit results
robust = oos_df[oos_df['sharpe_degradation'] < 0.4].copy()
print(f"\nRobust results (degradation < 40%): {len(robust)}")
if not robust.empty:
    print("\nTop 10 robust:")
    print(robust.head(10).to_string(index=False))

In [ ]:
# === Cell 10: Monte Carlo FTMO — Top Robust Performers ===

FTMO_ACCOUNT = 100_000
PROFIT_TARGET = 0.10
MAX_DAILY_LOSS = 0.05
MAX_TOTAL_LOSS = 0.10
N_SIMULATIONS = 10_000

# Use hourly frequency for trading days calculation
if TIMEFRAME == '1h':
    BARS_PER_DAY = 24  # crypto trades 24/7
    TRADING_DAYS = 30
    FTMO_BARS = TRADING_DAYS * BARS_PER_DAY
elif TIMEFRAME == '4h':
    BARS_PER_DAY = 6
    TRADING_DAYS = 30
    FTMO_BARS = TRADING_DAYS * BARS_PER_DAY

mc_results = []

# Take top 10 robust results
top_robust = robust.head(10) if not robust.empty else oos_df.head(5)

for idx, row in top_robust.iterrows():
    strategy = row['strategy']
    ticker = row['ticker']
    int_params = {k: int(row[k]) for k in PARAM_GRIDS[strategy].keys()}
    sl = row.get('sl_pct', None)
    tp = row.get('tp_pct', None)
    if pd.isna(sl): sl = None
    if pd.isna(tp): tp = None
    
    close = data[ticker]['close']
    high = data[ticker]['high']
    low = data[ticker]['low']
    
    regime = None
    if USE_REGIME_FILTER:
        regime = calc_regime_filter(close, high, low, VOLATILITY_LOOKBACK, VOLATILITY_THRESHOLD)
    
    # Generate signals on full sample
    if strategy == 'macd':
        entries, exits = macd_signals(close, int_params['fast'], int_params['slow'], int_params['signal'], regime)
    elif strategy == 'donchian':
        entries, exits = donchian_signals(close, high, low, int_params['entry'], int_params['exit'], int_params['filter'], regime)
    elif strategy == 'rsi_mr':
        entries, exits = rsi_mr_signals(close, int_params['rsi_len'], int_params['oversold'], int_params['overbought'], regime)
    elif strategy == 'stochastic':
        entries, exits = stochastic_signals(close, high, low, int_params['k_period'], int_params['d_period'], int_params['oversold'], int_params['overbought'], regime)
    
    e, x = apply_sl_tp(close, entries, exits, sl, tp)
    pf = vbt.Portfolio.from_signals(close=close, entries=e, exits=x,
                                    init_cash=INIT_CASH, fees=FEES, slippage=SLIPPAGE, freq=TIMEFRAME)
    
    # Get per-bar returns
    bar_rets = pf.returns().values.ravel()
    bar_rets = bar_rets[~np.isnan(bar_rets)]
    
    # Monte Carlo
    np.random.seed(42)
    n_passed = 0
    n_blown_total = 0
    n_blown_daily = 0
    finals = []
    
    for _ in range(N_SIMULATIONS):
        sim_rets = np.random.choice(bar_rets, size=FTMO_BARS, replace=True)
        eq = FTMO_ACCOUNT
        day_start_eq = FTMO_ACCOUNT
        
        for bar_idx, r in enumerate(sim_rets):
            # Reset daily tracking every BARS_PER_DAY bars
            if bar_idx % BARS_PER_DAY == 0:
                day_start_eq = eq
            
            eq *= (1 + r)
            
            # Check daily loss
            if (eq - day_start_eq) / FTMO_ACCOUNT < -MAX_DAILY_LOSS:
                n_blown_daily += 1
                break
            if (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT < -MAX_TOTAL_LOSS:
                n_blown_total += 1
                break
            if (eq - FTMO_ACCOUNT) / FTMO_ACCOUNT >= PROFIT_TARGET:
                n_passed += 1
                break
        finals.append(eq)
    
    pass_rate = n_passed / N_SIMULATIONS * 100
    if pass_rate >= 50: verdict = f"FAVORABLE"
    elif pass_rate >= 25: verdict = f"POSSIBLE"
    elif pass_rate >= 10: verdict = f"CHALLENGING"
    else: verdict = f"UNLIKELY"
    
    mc_results.append({
        'strategy': strategy, 'ticker': ticker,
        'params': str(int_params),
        'sl': sl, 'tp': tp,
        'oos_sharpe': row['oos_sharpe'],
        'pass_rate': pass_rate,
        'verdict': verdict,
        'blown_daily_pct': n_blown_daily / N_SIMULATIONS * 100,
        'blown_total_pct': n_blown_total / N_SIMULATIONS * 100,
        'median_equity': np.median(finals),
    })
    
    print(f"{strategy.upper():12s} {ticker:10s} | OOS Sharpe={row['oos_sharpe']:.3f} | "
          f"Pass={pass_rate:.1f}% ({verdict}) | Daily blow={n_blown_daily/N_SIMULATIONS*100:.0f}%")

mc_df = pd.DataFrame(mc_results)
print(f"\n{'='*60}")
print(f"Best FTMO candidates:")
print(mc_df.sort_values('pass_rate', ascending=False).head(10).to_string(index=False))

In [ ]:
# === Cell 11: Visualization — Top Performers ===

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(f'Crypto Intraday Grid Search Results — {TIMEFRAME} Timeframe', fontsize=16, fontweight='bold')

# (0,0) IS vs OOS Sharpe scatter
if not oos_df.empty:
    colors = {'macd': '#1f77b4', 'donchian': '#ff7f0e', 'rsi_mr': '#2ca02c', 'stochastic': '#d62728'}
    for strat in oos_df['strategy'].unique():
        mask = oos_df['strategy'] == strat
        axes[0, 0].scatter(oos_df.loc[mask, 'is_sharpe'], oos_df.loc[mask, 'oos_sharpe'],
                          c=colors.get(strat, 'gray'), label=strat.upper(), alpha=0.7, s=60)
    axes[0, 0].plot([oos_df['is_sharpe'].min(), oos_df['is_sharpe'].max()],
                    [oos_df['is_sharpe'].min(), oos_df['is_sharpe'].max()],
                    'k--', alpha=0.3, label='No degradation')
    axes[0, 0].set_xlabel('IS Sharpe', fontsize=11)
    axes[0, 0].set_ylabel('OOS Sharpe', fontsize=11)
    axes[0, 0].set_title('IS vs OOS Sharpe (above line = robust)', fontsize=13, fontweight='bold')
    axes[0, 0].legend(fontsize=9)
    axes[0, 0].grid(True, alpha=0.3)

# (0,1) Pass rate by strategy
if not mc_df.empty:
    mc_df_sorted = mc_df.sort_values('pass_rate', ascending=True)
    bar_colors = ['#2ca02c' if v >= 50 else '#ff7f0e' if v >= 25 else '#d62728' for v in mc_df_sorted['pass_rate']]
    axes[0, 1].barh(range(len(mc_df_sorted)), mc_df_sorted['pass_rate'], color=bar_colors)
    axes[0, 1].set_yticks(range(len(mc_df_sorted)))
    axes[0, 1].set_yticklabels([f"{r['strategy']} {r['ticker']}" for _, r in mc_df_sorted.iterrows()], fontsize=9)
    axes[0, 1].axvline(50, color='green', linestyle='--', alpha=0.5, label='50% threshold')
    axes[0, 1].axvline(25, color='orange', linestyle='--', alpha=0.5, label='25% threshold')
    axes[0, 1].set_xlabel('FTMO Pass Rate (%)', fontsize=11)
    axes[0, 1].set_title('Monte Carlo FTMO Pass Rates', fontsize=13, fontweight='bold')
    axes[0, 1].legend(fontsize=9)
    axes[0, 1].grid(axis='x', alpha=0.3)

# (1,0) Trade count distribution
if not oos_df.empty:
    axes[1, 0].hist(oos_df['oos_trades'], bins=30, color='#1f77b4', alpha=0.7, edgecolor='black')
    axes[1, 0].axvline(oos_df['oos_trades'].median(), color='red', linestyle='--', linewidth=2,
                       label=f"Median: {oos_df['oos_trades'].median():.0f}")
    axes[1, 0].set_xlabel('OOS Trades', fontsize=11)
    axes[1, 0].set_ylabel('Count', fontsize=11)
    axes[1, 0].set_title('Trade Frequency Distribution (OOS)', fontsize=13, fontweight='bold')
    axes[1, 0].legend(fontsize=10)
    axes[1, 0].grid(axis='y', alpha=0.3)

# (1,1) SL/TP heatmap for best strategy
if not oos_df.empty:
    best_strat = oos_df.iloc[0]['strategy']
    best_ticker = oos_df.iloc[0]['ticker']
    best_key = f"{best_strat}_{best_ticker}"
    if best_key in all_results:
        grid_df = all_results[best_key]
        # Pivot SL vs TP
        sl_vals = sorted(grid_df['sl_pct'].dropna().unique())
        tp_vals = sorted(grid_df['tp_pct'].dropna().unique())
        if len(sl_vals) > 1 and len(tp_vals) > 1:
            pivot = grid_df.groupby(['sl_pct', 'tp_pct'])['sharpe'].mean().reset_index()
            pivot_table = pivot.pivot(index='sl_pct', columns='tp_pct', values='sharpe')
            im = axes[1, 1].imshow(pivot_table.values, cmap='RdYlGn', aspect='auto')
            axes[1, 1].set_xticks(range(len(pivot_table.columns)))
            axes[1, 1].set_yticks(range(len(pivot_table.index)))
            axes[1, 1].set_xticklabels([f'{v:.0%}' if v else 'None' for v in pivot_table.columns], fontsize=8)
            axes[1, 1].set_yticklabels([f'{v:.0%}' if v else 'None' for v in pivot_table.index], fontsize=8)
            axes[1, 1].set_xlabel('Take-Profit', fontsize=11)
            axes[1, 1].set_ylabel('Stop-Loss', fontsize=11)
            axes[1, 1].set_title(f'SL/TP Sharpe Heatmap ({best_strat.upper()} {best_ticker})', fontsize=13, fontweight='bold')
            plt.colorbar(im, ax=axes[1, 1], shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# === Cell 12: Export Results + Telegram Notification ===

import json, os, urllib.request, urllib.parse

# Export
if os.path.exists('/content/drive/MyDrive'):
    export_root = '/content/drive/MyDrive/strategy_exports'
else:
    export_root = os.path.join(os.getcwd(), 'strategy_exports')

export_dir = os.path.join(export_root, 'Crypto_Intraday_Grid', TIMEFRAME, 'latest')
os.makedirs(export_dir, exist_ok=True)

# Save all grid results
for key, df in all_results.items():
    df.to_csv(os.path.join(export_dir, f'grid_{key}.csv'), index=False)

# Save OOS results
if not oos_df.empty:
    oos_df.to_csv(os.path.join(export_dir, 'oos_validation.csv'), index=False)

# Save MC results
if not mc_df.empty:
    mc_df.to_csv(os.path.join(export_dir, 'monte_carlo_ftmo.csv'), index=False)

# Save summary
summary = {
    'strategy': 'Crypto_Intraday_Grid',
    'timeframe': TIMEFRAME,
    'tickers': TICKERS,
    'strategies': STRATEGIES,
    'n_grid_results': {k: len(v) for k, v in all_results.items()},
    'n_oos_validated': len(oos_df) if not oos_df.empty else 0,
    'top_oos': oos_df.head(5).to_dict('records') if not oos_df.empty else [],
    'mc_results': mc_df.to_dict('records') if not mc_df.empty else [],
    'regime_filter': USE_REGIME_FILTER,
}
with open(os.path.join(export_dir, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Exported to: {export_dir}")

# Telegram
TG_BOT_TOKEN = os.getenv('TG_BOT_TOKEN', '')
TG_CHAT_ID = os.getenv('TG_CHAT_ID', '')

def send_telegram(message):
    url = f"https://api.telegram.org/bot{TG_BOT_TOKEN}/sendMessage"
    data = urllib.parse.urlencode({'chat_id': TG_CHAT_ID, 'text': message, 'parse_mode': 'Markdown'}).encode()
    try:
        req = urllib.request.Request(url, data=data)
        with urllib.request.urlopen(req, timeout=10) as resp:
            return resp.status == 200
    except Exception as e:
        print(f"Telegram failed: {e}")
        return False

msg = f"*Crypto Intraday Grid Search Complete*\n\nTimeframe: {TIMEFRAME}\nTickers: {', '.join(TICKERS)}\n\n"
if not oos_df.empty:
    msg += f"*Top 5 OOS Results:*\n"
    for _, r in oos_df.head(5).iterrows():
        msg += f"  {r['strategy'].upper()} {r['ticker']}: OOS Sharpe={r['oos_sharpe']:.3f}, Trades={r['oos_trades']}\n"
if not mc_df.empty:
    best_mc = mc_df.sort_values('pass_rate', ascending=False).iloc[0]
    msg += f"\n*Best FTMO:*\n  {best_mc['strategy'].upper()} {best_mc['ticker']}: {best_mc['pass_rate']:.1f}% pass rate ({best_mc['verdict']})"

if send_telegram(msg):
    print("Results sent to Telegram!")
else:
    print("Telegram failed. Message:")
    print(msg)